# Transfer Learning using PyTorch

# Import Libraries

In [47]:
import pandas as pd
import numpy as np

In [48]:
from sklearn.model_selection import train_test_split

In [49]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

In [50]:
import kagglehub
import os

# Import Data

In [51]:
path = kagglehub.dataset_download("zalando-research/fashionmnist")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fashionmnist' dataset.
Path to dataset files: /kaggle/input/fashionmnist


In [52]:
df = pd.read_csv(f'{path}/fashion-mnist_train.csv')
df.shape

(60000, 785)

In [53]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Check `GPU` availability

In [54]:
device = 'cpu'
x = torch.rand(2, 3)

# Move the tensor to CUDA (if available)
if torch.cuda.is_available():
  device = torch.device("cuda")
  x_cuda = x.to(device)
  print(f"Tensor on: {x_cuda.device}") # Output: Tensor on: cuda:0

device

Tensor on: cuda:0


device(type='cuda')

# Extract Features and Labels

In [55]:
X = df.drop('label', axis=1)
X = X.values
y = df['label'].values

In [56]:
X[0][100:150]

array([136,  61,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,  88, 201, 228, 225, 255, 115,  62,
       137, 255, 235, 222, 255, 135,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,  47, 252, 234, 238, 224])

# Train Test Split

In [57]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=17)

X_train.shape, X_test.shape

((48000, 784), (12000, 784))

# Prepare Data for `VGG16`

```
1. Reshape to (28, 28)
2. Convert Datatype to (uint8) # Usefull for PIL
3. Covert 1D-3D (Gray-RGB image)
4. Make Tensor from PIL image

5. Resize (256, 256) # For PIL image
6. Center Crop (224, 244)
7. Tensor Scale and Convert # 0-1 range
8. Normalized using Predefine Mu and Sigma
```

## Transformes

In [58]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# `Dataset` Class

In [59]:
class CustomDataset(Dataset):
  def __init__(self, features, labels, transform=None):
    self.features = features
    self.labels= labels
    self.transform = transform

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, index):

    # resize to (28, 28)
    image = self.features[index].reshape(28, 28)

    # change datatype to np.uint8
    image = image.astype(np.uint8)

    # change B&W to RGB -> (H,W,C) -> (C,H,W)
    image = np.stack([image]*3, axis=-1)

    # convert array to PIL image
    image = Image.fromarray(image)

    # apply transforms
    image = self.transform(image)

    # return
    return image, torch.tensor(self.labels[index], dtype=torch.long)


In [70]:
train_dataset = CustomDataset(features=X_train, labels=y_train, transform=transform)
test_dataset = CustomDataset(features=X_test, labels=y_test, transform=transform)

# DataLoader

In [71]:
train_loader = DataLoader(
    dataset = train_dataset,
    batch_size = 256,
    shuffle= True,
    pin_memory=True
  )

In [72]:
test_loader = DataLoader(
    dataset = test_dataset,
    batch_size = 226,
    shuffle= True,
    pin_memory=True
  )

# Load `VGG16` Model

In [73]:
vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

## Freez Conv Params

In [74]:
for param in vgg16.features.parameters():
  param.requires_grad = False

# Custom Model

In [75]:
vgg16.classifier = nn.Sequential(
    # Hidden layer
    nn.Linear(25088, 1024),   # smaller than 2048
    nn.BatchNorm1d(1024),
    nn.ReLU(),                 # ReLU is simpler than SELU
    nn.Dropout(0.5),           # moderate dropout

    # Second hidden layer
    nn.Linear(1024, 256),      # smaller than 512
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.3),

    # Output layer
    nn.Linear(256, 10)
)

In [76]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

## Set Learning Rate and Epochs

In [77]:
learning_rate = 0.0001
epochs = 7

In [78]:
# model Device Changes
vgg16 = vgg16.to(device)

# Initiate Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(params=vgg16.parameters(), lr=learning_rate, weight_decay=1e-4)

## Training Pipeline

In [79]:
for epoch in range(epochs):
  losses = []

  vgg16.train()

  for batch_features, batch_labels in  train_loader:

    # move data to gpu
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    # forward pass
    y_pred = vgg16(batch_features)

    # loss calculate
    loss = criterion(y_pred, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update params
    optimizer.step()

    # store losses
    losses.append(loss.item())

  avg_loss = np.mean(losses)
  print(f'Epochs: {epoch+1}, Loss: {avg_loss}')

Epochs: 1, Loss: 0.4459123322145736
Epochs: 2, Loss: 0.2193777504119467
Epochs: 3, Loss: 0.1462058056383691
Epochs: 4, Loss: 0.09892926711906144
Epochs: 5, Loss: 0.06689598330078607
Epochs: 6, Loss: 0.044357106664237825
Epochs: 7, Loss: 0.03574599199829266


# Evaluation

## Set `eval` Model (Crucial)

In [81]:
vgg16.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

## Eval Code

In [82]:
total_rows = 0
correct = 0

with torch.no_grad():
  for features, labels in test_loader:

      # move data to gpu
      features = features.to(device)
      labels = labels.to(device)

      # Output from network: [batch, 10]
      y_pred = vgg16(features)

      # Predicted class index
      _, predict = torch.max(y_pred, 1)

      # Count correct predictions
      correct += (predict == labels).sum().item()
      total_rows += labels.size(0)

  avg_accuracy = correct / total_rows
  print(f'Average Accuracy: {avg_accuracy:.4f}')


Average Accuracy: 0.9282


In [83]:
total_rows = 0
correct = 0

with torch.no_grad():
  for features, labels in train_loader:

      # move data to gpu
      features = features.to(device)
      labels = labels.to(device)

      # Output from network: [batch, 10]
      y_pred = vgg16(features)

      # Predicted class index
      _, predict = torch.max(y_pred, 1)

      # Count correct predictions
      correct += (predict == labels).sum().item()
      total_rows += labels.size(0)

  avg_accuracy = correct / total_rows
  print(f'Average Accuracy: {avg_accuracy:.4f}')

Average Accuracy: 0.9979
